# Анализ и оформление результатов

## 1. Импорты и настройка

In [1]:
import json, os, subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs('data/processed', exist_ok=True)
plt.style.use('seaborn-v0_8-darkgrid')

## 2. Загрузка метрик и создание сводной таблицы

In [2]:
# Загружаем метрики
with open('data/processed/all_models_metrics.json', 'r') as f:
    metrics = json.load(f)

# Создаём DataFrame
df_metrics = pd.DataFrame(metrics).T    # транспонируем, чтобы модели были строками
df_metrics.index.name = 'Model'

# Округляем для читаемости
df_metrics = df_metrics.round(4)

print("Сводная таблица метрик:")
display(df_metrics)

# Сохраняем в CSV
df_metrics.to_csv('data/processed/comparison_table.csv')

# Находим лучшую модель по R²
best_model = df_metrics['R2'].idxmax()
best_r2 = df_metrics.loc[best_model, 'R2']
print(f"\nЛучшая модель по R²: **{best_model}** (R² = {best_r2})")

Сводная таблица метрик:


,MSE,MAE,R2
Model,,,
linear_regression,0.2284,0.2867,0.7237
decision_tree,0.1712,0.2309,0.7928
catboost,0.1707,0.2553,0.7935
xgboost,0.1613,0.1998,0.8048
neural_network,0.1725,0.2136,0.7913



Лучшая модель по R²: **xgboost** (R² = 0.8048)


## 3. Генерация графа DVC

In [3]:
# Попробуем сгенерировать граф DVC с помощью встроенной команды
dag_path = 'data/processed/dag.png'
try:
    # Генерируем DOT-описание и передаём в Graphviz (требуется установленный Graphviz)
    dot = subprocess.check_output([sys.executable, '-m', 'dvc', 'dag', '--dot'], 
                                  text=True, stderr=subprocess.STDOUT)
    # Сохраняем в PNG через graphviz Python-библиотеку
    from graphviz import Source
    s = Source(dot, format='png')
    s.render(filename='dag', directory='data/processed', cleanup=True)
    print(f"Граф DVC сохранён в {dag_path}")
except Exception as e:
    print(f"Не удалось построить изображение графа ({e}).")
    print("Вы можете выполнить команду `dvc dag` в терминале и прикрепить скриншот.")

Граф DVC сохранён в data/processed/dag.png


## 4. Вывод по работе

In [4]:
print("=== ВЫВОД ПО ЛАБОРАТОРНОЙ РАБОТЕ ===\n")

print("1. Проведён разведочный анализ данных (EDA). Определены ключевые факторы, влияющие на стоимость страховки: "
      "курение, возраст, BMI. Целевая переменная имеет правостороннюю асимметрию, поэтому было применено "
      "логарифмическое преобразование.")

print("2. Данные разделены на три выборки: train, validation и test. "
      "Для обучения использовались пять моделей: линейная регрессия, дерево решений, CatBoost, XGBoost и "
      "нейронная сеть (MLPRegressor).")

print("3. Метрики качества (MSE, MAE, R²) вычислены на тестовой выборке и сведены в таблицу. "
      "Лучшие результаты показали градиентные бустинги (CatBoost / XGBoost), так как они способны "
      "улавливать нелинейные зависимости и взаимодействия признаков. Дерево решений также показало "
      "неплохой результат, но уступает ансамблевым методам. Линейная регрессия, как ожидалось, "
      "имеет самую низкую точность из-за нелинейности данных. Нейронная сеть показала сравнимые "
      "с бустингами результаты, но требует больше вычислительных ресурсов и настройки.")

print("4. DVC обеспечил воспроизводимость всех этапов: от подготовки данных до оценки моделей. "
      "Все параметры зафиксированы в params.yaml, что позволяет легко повторять эксперименты.")

print("\nРекомендация: для прогнозирования медицинских расходов целесообразно использовать CatBoost или XGBoost, "
      "как наиболее точные и интерпретируемые модели в данной задаче.")

=== ВЫВОД ПО ЛАБОРАТОРНОЙ РАБОТЕ ===

1. Проведён разведочный анализ данных (EDA). Определены ключевые факторы, влияющие на стоимость страховки: курение, возраст, BMI. Целевая переменная имеет правостороннюю асимметрию, поэтому было применено логарифмическое преобразование.
2. Данные разделены на три выборки: train, validation и test. Для обучения использовались пять моделей: линейная регрессия, дерево решений, CatBoost, XGBoost и нейронная сеть (MLPRegressor).
3. Метрики качества (MSE, MAE, R²) вычислены на тестовой выборке и сведены в таблицу. Лучшие результаты показали градиентные бустинги (CatBoost / XGBoost), так как они способны улавливать нелинейные зависимости и взаимодействия признаков. Дерево решений также показало неплохой результат, но уступает ансамблевым методам. Линейная регрессия, как ожидалось, имеет самую низкую точность из-за нелинейности данных. Нейронная сеть показала сравнимые с бустингами результаты, но требует больше вычислительных ресурсов и настройки.
4. DVC о